# FaceCloud Private Model Training (Google Colab)

This notebook trains `FaceCloudCurveNet` using the low-light images stored inside `facecloud_backend/Dataset/image`. Checkpoints are written to Google Drive so an interrupted Colab session can be resumed.

**Before running:** choose **Runtime → Change runtime type → T4 GPU**. Keep the dataset and generated weights private.

## 1. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. Configure paths

The cell checks common locations automatically. If your folder is elsewhere, edit `BACKEND_DIR` only.

In [ ]:
from pathlib import Path

BACKEND_CANDIDATES = [
    Path('/content/drive/MyDrive/facecloud_backend'),
    Path('/content/drive/MyDrive/FaceCloud/facecloud_backend'),
    Path('/content/facecloud_backend'),
]
BACKEND_DIR = next((path for path in BACKEND_CANDIDATES if path.is_dir()), None)
if BACKEND_DIR is None:
    raise FileNotFoundError(
        'facecloud_backend was not found. Upload it to MyDrive or edit BACKEND_CANDIDATES.'
    )

DATASET_DIR = BACKEND_DIR / 'Dataset'
TRAINER = BACKEND_DIR / 'scripts' / 'train_facecloud_curve_model.py'
OUTPUT_DIR = Path('/content/drive/MyDrive/FaceCloud/private-model-v1')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

assert (DATASET_DIR / 'image').is_dir(), f'Missing dataset image folder: {DATASET_DIR / "image"}'
assert TRAINER.is_file(), f'Missing trainer: {TRAINER}'
print('Backend:', BACKEND_DIR)
print('Dataset:', DATASET_DIR)
print('Outputs:', OUTPUT_DIR)

## 3. Verify the GPU

In [ ]:
import subprocess
import torch

subprocess.run(['nvidia-smi'], check=False)
assert torch.cuda.is_available(), (
    'GPU is not active. Select Runtime > Change runtime type > T4 GPU, then reconnect.'
)
print('PyTorch:', torch.__version__)
print('GPU:', torch.cuda.get_device_name(0))
print('CUDA:', torch.version.cuda)

## 4. Install training and export dependencies

In [ ]:
import sys
import subprocess

subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q',
    'onnx>=1.16', 'onnxruntime>=1.18', 'Pillow>=10', 'numpy>=1.26'
], check=True)
print('Dependencies installed.')

## 5. Audit the input images

In [ ]:
from PIL import Image

image_files = sorted((DATASET_DIR / 'image').glob('*.png'))
assert image_files, 'No PNG files were found in Dataset/image.'
corrupt = []
for path in image_files:
    try:
        with Image.open(path) as image:
            image.verify()
    except Exception as error:
        corrupt.append((str(path), str(error)))

print('Images found:', len(image_files))
print('Corrupt images:', len(corrupt))
assert not corrupt, f'Corrupt images must be removed before training: {corrupt[:5]}'

## 6. Training configuration

The defaults use a 256×256 crop, batch size 16 and 100 epochs. If Colab reports insufficient GPU memory, change `BATCH_SIZE` to 8.

In [ ]:
IMAGE_SIZE = 256
BATCH_SIZE = 16
EPOCHS = 100
LEARNING_RATE = 1e-4
WORKERS = 2
SEED = 2026
EXPOSURE_TARGET = 0.60

print({
    'image_size': IMAGE_SIZE, 'batch_size': BATCH_SIZE, 'epochs': EPOCHS,
    'learning_rate': LEARNING_RATE, 'workers': WORKERS,
    'seed': SEED, 'exposure_target': EXPOSURE_TARGET,
})

## 7. Train or resume

If `last.pt` exists, this cell resumes automatically. Otherwise it starts a new run. Do not run two training sessions against the same output folder.

In [ ]:
import sys
import subprocess

command = [
    sys.executable, str(TRAINER),
    '--dataset', str(DATASET_DIR),
    '--output-dir', str(OUTPUT_DIR),
    '--image-size', str(IMAGE_SIZE),
    '--batch-size', str(BATCH_SIZE),
    '--epochs', str(EPOCHS),
    '--learning-rate', str(LEARNING_RATE),
    '--workers', str(WORKERS),
    '--seed', str(SEED),
    '--exposure-target', str(EXPOSURE_TARGET),
]
resume_checkpoint = OUTPUT_DIR / 'last.pt'
if resume_checkpoint.is_file():
    command.extend(['--resume', str(resume_checkpoint)])
    print('Resuming from:', resume_checkpoint)
else:
    print('Starting a new training run.')

print('Command:', ' '.join(command))
subprocess.run(command, check=True)

## 8. Plot the training history

In [ ]:
import json
import matplotlib.pyplot as plt

history_path = OUTPUT_DIR / 'history.jsonl'
records = [json.loads(line) for line in history_path.read_text().splitlines() if line.strip()]
epochs = [record['epoch'] for record in records]
train_loss = [record['train']['loss'] for record in records]
validation_loss = [record['validation']['loss'] for record in records]

plt.figure(figsize=(10, 5))
plt.plot(epochs, train_loss, label='Training loss')
plt.plot(epochs, validation_loss, label='Validation loss')
plt.xlabel('Epoch')
plt.ylabel('Total loss')
plt.title('FaceCloudCurveNet training history')
plt.grid(alpha=0.25)
plt.legend()
plt.show()
print('Best validation loss:', min(validation_loss))

## 9. Validate the exported ONNX model

In [ ]:
import numpy as np
import onnx
import onnxruntime as ort

onnx_path = OUTPUT_DIR / 'facecloud_curve_best.onnx'
assert onnx_path.is_file(), f'ONNX model was not generated: {onnx_path}'
model = onnx.load(str(onnx_path))
onnx.checker.check_model(model)
session = ort.InferenceSession(str(onnx_path), providers=['CPUExecutionProvider'])
test_input = np.random.rand(1, 3, 128, 128).astype(np.float32)
test_output = session.run(None, {'image': test_input})[0]
assert test_output.shape == (1, 24, 128, 128), test_output.shape
assert np.isfinite(test_output).all(), 'ONNX output contains invalid values.'
print('ONNX validation passed:', onnx_path)
print('Output shape:', test_output.shape)

## 10. Review outputs

The private outputs remain in `MyDrive/FaceCloud/private-model-v1`. Do not publish `best.pt`, `last.pt`, or the ONNX file. The next required step is a held-out visual and quantitative evaluation before production deployment.

In [ ]:
for path in sorted(OUTPUT_DIR.iterdir()):
    if path.is_file():
        print(f'{path.name:35s} {path.stat().st_size / (1024 * 1024):8.2f} MB')